# Fase 5 - Avaliação dos Modelos

Integrantes: Gabriel Casagrande, Guilherme Carrer, Filipe Guizzo, Pedro Canto

*Tema:* Análise dos Gastos Públicos com Servidores da Prefeitura de Criciúma

Depois de construir o modelo na Fase 4, chegou a hora de parar e olhar com calma pra esse resultado antes de sair publicando ele por aí. O objetivo aqui não é só olhar pro número do R² e dizer "deu bom" ou "deu ruim", mas sim entender se esse modelo realmente responde à pergunta que a gente fez no começo do projeto e se algum detalhe importante passou batido nas fases anteriores.

## 5.1 Avaliar os Resultados

### 5.1.1 Tarefa

Na Fase 4 a gente já tinha calculado as métricas técnicas do modelo (RMSE e R²), então aqui o foco muda um pouco: precisamos olhar pra esses números do ponto de vista do objetivo de negócio que definimos na Fase 1, que era entender como os gastos com servidores estão distribuídos e se dá pra identificar algum padrão de alocação de recursos a partir do vínculo empregatício.

Pra isso, recarregamos a base e refizemos o pipeline completo (limpeza, Box-Cox e treino do CatBoost), só pra garantir que estamos avaliando exatamente o mesmo modelo que documentamos antes.

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import warnings
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from catboost import CatBoostRegressor

warnings.filterwarnings('ignore')

df = pd.read_csv('dados_governo.csv')

df['remuneracao'] = df['remuneraaao_contratual_r$'].str.replace(r'R\$', '', regex=True)
df['remuneracao'] = df['remuneracao'].str.replace(' ', '', regex=False)
df['remuneracao'] = df['remuneracao'].str.replace('.', '', regex=False)
df['remuneracao'] = df['remuneracao'].str.replace(',', '.', regex=False)
df['remuneracao'] = pd.to_numeric(df['remuneracao'], errors='coerce')

df = df.dropna(subset=['remuneracao'])
df = df[df['remuneracao'] > 0]

media = df['remuneracao'].mean()
desvio_padrao = df['remuneracao'].std()

df['remuneracao_boxcox'], lambda_opt = stats.boxcox(df['remuneracao'])

X = df[['vanculo_empregatacio']]
y = df['remuneracao_boxcox']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = CatBoostRegressor(iterations=500, learning_rate=0.1, depth=6, verbose=0)
model.fit(X_train, y_train, cat_features=[0])

y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
importances = model.get_feature_importance()

print(f"Linhas usadas no modelo: {len(df)}")
print(f"Lambda do Box-Cox: {lambda_opt:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²: {r2:.4f}")
print(f"Importância do vínculo empregatício: {importances[0]:.2f}%")

Linhas usadas no modelo: 17604
Lambda do Box-Cox: -0.0571
RMSE: 0.3144
R²: 0.6833
Importância do vínculo empregatício: 100.00%


### 5.1.2 Saída

#### 5.1.2.1 Avaliação do Modelo em Razão dos Critérios de Sucesso da Área de Negócio

Voltando lá na Fase 1, um dos critérios de sucesso era "identificar padrões relevantes nos gastos com servidores" e "fornecer insights que ajudem na interpretação dos dados". Olhando o modelo final com esse filtro:

- O R² ficou em **0.6833**, ou seja, o vínculo empregatício sozinho consegue explicar cerca de **68%** da variação da remuneração (já na escala Box-Cox). Pra um modelo que usa só uma variável de entrada, esse é um resultado relevante e mostra que o tipo de vínculo realmente pesa na composição do salário.
- O RMSE de **0.3144** mostra que o erro do modelo é pequeno em relação à dispersão da variável transformada, então o CatBoost não tá só "chutando" valores aleatórios, ele aprendeu alguma relação real.
- A importância da variável apareceu como 100%, mas isso é meio óbvio porque só usamos uma feature — não dá pra comparar o peso do vínculo contra outras variáveis porque elas não entraram no modelo.

Então, em relação ao critério de sucesso definido no início do projeto, dá pra dizer que o modelo cumpre o que foi proposto: ele confirma estatisticamente que existe um padrão de alocação ligado ao vínculo, que era basicamente a pergunta-problema do grupo. Só que um R² de 68.33% também deixa claro que tem mais de 30% da variação salarial que não está sendo explicada só pelo vínculo, e isso provavelmente vem de outras colunas da base (cargo, carga horária, órgão) que não entraram no modelo.

#### 5.1.2.2 Modelos Aprovados

A gente decidiu aprovar o modelo CatBoost (vínculo → remuneração Box-Cox) **com algumas ressalvas**, e não como um modelo perfeito e definitivo. Os motivos:

1. Quando fomos checar de novo a variável de vínculo empregatício pra essa avaliação, reparamos que **9253 registros (cerca de 53% da base)** estão marcados com o valor `"null"` direto na planilha — não é um campo vazio do pandas, é um texto mesmo escrito assim na fonte de dados. Isso passou meio batido nas fases anteriores e é bastante gente sem vínculo identificado de fato.
2. O modelo usa só uma variável categórica. Isso fecha com a pergunta do projeto (que é focada no vínculo), mas limita bastante o quanto o modelo consegue explicar no geral.
3. Mesmo depois do Box-Cox, os testes de normalidade (KS e Shapiro) continuaram dando p-value bem abaixo de 0.05, então tecnicamente a distribuição não fica "normal" — só menos enviesada. Isso já tinha sido discutido na Fase 3 e segue valendo aqui.

Resumindo: aprovamos o modelo pra seguir pra publicação, mas com a ressalva clara sobre os registros `"null"` e sobre o fato de ser um modelo de uma variável só.

## 5.2 Processo de Revisão

### 5.2.1 Tarefa

Com o modelo já avaliado, vale dar uma olhada geral em todo o processo, desde a Fase 1 até aqui, pra ver se alguma coisa importante ficou de lado no meio do caminho.

In [2]:
print(df['vanculo_empregatacio'].value_counts())
print()
qtd_null = df['vanculo_empregatacio'].str.strip().eq('null').sum()
print(f"Registros com vínculo 'null': {qtd_null} ({qtd_null/len(df)*100:.2f}% da base)")

vanculo_empregatacio
 null                                            9253
 Contrato Adm Temporário                         5216
 Estatutário                                     2106
 Servidor Público não Efetivo                     748
 Celetista                                        172
 Conselheiro Tutelar                                54
 Agente Político                                    50
 Recebido por cessão em cargo efetivo                2
 Aposentados e Pensionistas                          1
 Recebido por cessão para cargo em comissão          1
 Vinculo Temporario SEM SEFIP                        1
Name: count, dtype: int64

Registros com vínculo 'null': 9253 (52.56% da base)


### 5.2.2 Saída

#### 5.2.2.1 Processo de Revisão

Revendo o caminho que o projeto percorreu até aqui:

- Na **Fase 1**, a gente definiu como objetivo entender a distribuição dos gastos com servidores e identificar padrões de alocação. Isso ficou bem definido e foi seguido até o final.
- Na **Fase 3 (preparação dos dados)**, fizemos a limpeza da coluna de remuneração e aplicamos os testes de normalidade e o Box-Cox. Esse trabalho foi sólido, só que naquele momento não reparamos a fundo na qualidade da coluna `vanculo_empregatacio`, focamos mais na remuneração em si.
- Na **Fase 4 (modelagem)**, treinamos o CatBoost usando o vínculo como única variável e chegamos no R² de 68.33%. O modelo em si foi bem construído (split treino/teste, parâmetros razoáveis), mas não fizemos validação cruzada nem testamos outras variáveis como entrada.
- Já aqui na **Fase 5**, ao reavaliar tudo, é que apareceu o problema dos 9253 registros `"null"` — mais da metade da base. Isso não invalida o modelo, mas mostra que a etapa de entendimento dos dados (Fase 2) poderia ter detectado esse ponto antes.

De forma geral, o processo seguiu a lógica do CRISP-DM corretamente, fase por fase, e o modelo final é estatisticamente válido. O que ficou de lição é que a qualidade de uma variável categórica usada como entrada do modelo merece uma checagem mais cedo no processo, e não só na hora de avaliar o resultado final.

## 5.3 Determinar os próximos passos

### 5.3.1 Tarefa

Com a avaliação e a revisão feitas, o grupo precisa decidir como seguir: encerrar o projeto aqui, continuar melhorando o modelo, ou abrir frente pra um projeto novo a partir do que foi descoberto.

### 5.3.2 Saída

#### 5.3.2.1 Lista de Possíveis Ações

1. Encerrar esse ciclo do projeto publicando o modelo atual (R² = 68.33%), deixando registrada a ressalva sobre os registros `"null"`.
2. Voltar pra fase de preparação dos dados e tratar (ou remover) os 9253 registros sem vínculo identificado, pra ver se o R² melhora.
3. Incluir outras colunas da base (cargo, órgão, carga horária) no modelo, pra tentar explicar a parte da variação que o vínculo sozinho não explica.
4. Rodar uma validação cruzada (k-fold) em vez de um único split de treino e teste, pra confirmar se o RMSE e o R² se mantêm estáveis.
5. Encerrar o projeto da disciplina aqui mesmo, já que as fases 1 a 6 do CRISP-DM foram cumpridas, e deixar as ações 2, 3 e 4 como sugestão pra quem continuar o trabalho depois.

#### 5.3.2.2 Decisão

O grupo optou por seguir com as ações 1 e 5: finalizar esse ciclo do projeto com o modelo atual, já que ele responde de forma satisfatória à pergunta-problema definida na Fase 1, e os prazos da disciplina estão batendo. As ações 2, 3 e 4 vão ficar registradas na Fase 6 como recomendações pra uma possível continuação do projeto.